In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../"))
from src.dataset.data_loader import load_competition_data

train_dataset, test_dataset, sample_submission = load_competition_data()
print("Train shape:", train_dataset.shape)
print("Test shape :", test_dataset.shape)

In [ ]:
import os
import numpy as np
import torch, warnings, logging
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, TaskType
from tqdm import tqdm

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OPTIONS = ['A', 'B', 'C', 'D', 'E']
print(f"Device : {DEVICE}")

In [ ]:
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

from datasets.utils.logging import disable_progress_bar as ds_disable_progress_bar
ds_disable_progress_bar()

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)

In [ ]:
def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    if actual not in predicted:
        return 0.0
    return 1.0 / (predicted.index(actual) + 1)

def mapk(df, pred_col):
    return df.apply(lambda row: apk(row['answer'], row[pred_col].split()), axis=1).mean()

In [ ]:
label_to_id = {opt: i for i, opt in enumerate(OPTIONS)}
id_to_label = {i: opt for opt, i in label_to_id.items()}

print("Label mapping (Milestone 5 setup):")
for i, opt in id_to_label.items():
    print(f"  Label ID {i}  ->  Option {opt}")

def format_seqcls(row):
    """Pack prompt + all 5 options into a single sequence-classification input."""
    parts = [str(row['prompt'])]
    for opt in OPTIONS:
        parts.append(f"{opt}) {row[opt]}")
    return " [SEP] ".join(parts)

demo_text = format_seqcls(train_dataset.loc[0])
print(" ")
print("Row 0 formatted sequence-classification input:")
print(demo_text)

In [ ]:
M5_train_df = train_dataset.copy()
M5_train_df['label'] = M5_train_df['answer'].map(label_to_id)

prompt_answer_m5 = M5_train_df.groupby('prompt')['answer'].first()
unique_prompts_m5 = prompt_answer_m5.index.astype(str).to_numpy()

train_prompts_m5, val_prompts_m5 = train_test_split(
    unique_prompts_m5, test_size=0.15, random_state=SEED, stratify=prompt_answer_m5.values
)

m5_train_split = M5_train_df[M5_train_df['prompt'].isin(train_prompts_m5)].reset_index(drop=True)
m5_val_split = M5_train_df[M5_train_df['prompt'].isin(val_prompts_m5)].reset_index(drop=True)

print(m5_train_split.shape)
print(m5_val_split.shape)

In [ ]:
def build_seqcls_dataset(df, tokenizer, max_length=256, has_labels=True):
    texts = [format_seqcls(row) for _, row in df.iterrows()]
    encoded = tokenizer(texts, padding="max_length", truncation=True, max_length=max_length)
    records = []
    for i in range(len(df)):
        rec = {"input_ids": encoded["input_ids"][i], "attention_mask": encoded["attention_mask"][i]}
        if has_labels:
            rec["labels"] = int(df.iloc[i]["label"])
        records.append(rec)
    return Dataset.from_list(records)

def seqcls_compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    accuracy = (preds == labels).mean()
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    top3 = np.argsort(-probs, axis=1)[:, :3]
    map3_scores = []
    for rank_row, true_label in zip(top3, labels):
        rank_row = list(rank_row)
        map3_scores.append(1.0 / (rank_row.index(true_label) + 1) if true_label in rank_row else 0.0)
    return {"accuracy": accuracy, "map3": float(np.mean(map3_scores))}

In [ ]:
import sys
!{sys.executable} -m pip install sentencepiece protobuf tiktoken

In [ ]:
import sentencepiece
print("sentencepiece OK:", sentencepiece.__version__)

In [ ]:
DEBERTA_MODEL = "microsoft/deberta-v3-small"
DEBERTA_MAX_LEN = 256

deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL)
hf_m5_train_deberta = build_seqcls_dataset(m5_train_split, deberta_tokenizer, max_length=DEBERTA_MAX_LEN)
hf_m5_val_deberta = build_seqcls_dataset(m5_val_split, deberta_tokenizer, max_length=DEBERTA_MAX_LEN)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_MODEL, num_labels=5, torch_dtype=torch.float32
).float().to(DEVICE)

deberta_training_args = TrainingArguments(
    output_dir="./m5_deberta",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    logging_steps=20,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

deberta_trainer = Trainer(
    model=deberta_model,
    args=deberta_training_args,
    train_dataset=hf_m5_train_deberta,
    eval_dataset=hf_m5_val_deberta,
    data_collator=default_data_collator,
    compute_metrics=seqcls_compute_metrics,
)

deberta_train_result = deberta_trainer.train()
print(deberta_train_result)

# load_best_model_at_end=True => deberta_trainer.model is already the best checkpoint, in memory
deberta_val_metrics = deberta_trainer.evaluate()
print("DeBERTa final validation metrics:", deberta_val_metrics)

In [ ]:
ROBERTA_MODEL = "roberta-base"
ROBERTA_MAX_LEN = 256

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
hf_m5_train_roberta = build_seqcls_dataset(m5_train_split, roberta_tokenizer, max_length=ROBERTA_MAX_LEN)
hf_m5_val_roberta = build_seqcls_dataset(m5_val_split, roberta_tokenizer, max_length=ROBERTA_MAX_LEN)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_MODEL, num_labels=5, torch_dtype=torch.float32
).float().to(DEVICE)

roberta_training_args = TrainingArguments(
    output_dir="./m5_roberta",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    logging_steps=20,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,
    train_dataset=hf_m5_train_roberta,
    eval_dataset=hf_m5_val_roberta,
    data_collator=default_data_collator,
    compute_metrics=seqcls_compute_metrics,
)

roberta_train_result = roberta_trainer.train()
print(roberta_train_result)

roberta_val_metrics = roberta_trainer.evaluate()
print("RoBERTa final validation metrics:", roberta_val_metrics)

In [ ]:
deberta_ft_tokenizer = deberta_tokenizer
deberta_ft_model = deberta_trainer.model.to(DEVICE).eval()

roberta_ft_tokenizer = roberta_tokenizer
roberta_ft_model = roberta_trainer.model.to(DEVICE).eval()

def get_probs(text, tokenizer, model, max_length=256):
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=max_length, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    return F.softmax(logits, dim=-1).squeeze(0).cpu()

def probs_batch(texts, tokenizer, model, max_length=256):
    enc = tokenizer(texts, truncation=True, padding="max_length", max_length=max_length, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    return F.softmax(logits, dim=-1).cpu()

print("Trained DeBERTa and RoBERTa models ready for Q/A (in-memory, no disk reload).")

# ============================================================
# Q1 — DeBERTa top prediction + probability, row index 25
# ============================================================
row25 = m5_val_split.iloc[25]
text25 = format_seqcls(row25)
print("Row 25 prompt:", row25['prompt'])
print("True answer  :", row25['answer'])

deberta_probs_25 = get_probs(text25, deberta_ft_tokenizer, deberta_ft_model, DEBERTA_MAX_LEN)
roberta_probs_25 = get_probs(text25, roberta_ft_tokenizer, roberta_ft_model, ROBERTA_MAX_LEN)

q1_top_id = int(torch.argmax(deberta_probs_25))
q1_answer = f"{id_to_label[q1_top_id]}, probability of {id_to_label[q1_top_id]} = {deberta_probs_25[q1_top_id]:.4f}"
print(f"\nQ1 — {q1_answer}")
print(" ")

# ============================================================
# Q2 — simple probability averaging, row index 25
# Average Probability = [P(DeBERTa) + P(RoBERTa)] / 2
# ============================================================
avg_probs_25 = (deberta_probs_25 + roberta_probs_25) / 2
q2_top_id = int(torch.argmax(avg_probs_25))
q2_answer = id_to_label[q2_top_id]
print(f"Q2 — highest averaged-probability option: {q2_answer}  (p={avg_probs_25[q2_top_id]:.4f})")
print(" ")

# ============================================================
# Q3 — weighted probability averaging, row index 25
# P(final) = 0.7 * P(DeBERTa) + 0.3 * P(RoBERTa)
# ============================================================
W_DEBERTA, W_ROBERTA = 0.70, 0.30

weighted_probs_25 = W_DEBERTA * deberta_probs_25 + W_ROBERTA * roberta_probs_25
q3_top_id = int(torch.argmax(weighted_probs_25))
q3_answer = id_to_label[q3_top_id]
print(f"Q3 — top option after weighted ensembling: {q3_answer}  (p={weighted_probs_25[q3_top_id]:.4f})")
print(" ")

# ============================================================
# Q4 — Top-3 Kaggle-format prediction string, row index 25
# ============================================================
top3_ids_25 = torch.argsort(weighted_probs_25, descending=True)[:3].tolist()
q4_answer = " ".join(id_to_label[i] for i in top3_ids_25)
print(f"Q4 — Top-3 prediction string for row 25: {q4_answer}")
print(" ")

# ============================================================
# Q5 — run the weighted ensemble on every row of test.csv,
#      save submission.csv in the required Kaggle format
# ============================================================
test_df_m5 = test_dataset.copy()
test_texts_m5 = [format_seqcls(row) for _, row in test_df_m5.iterrows()]

test_top3_preds = []
BATCH = 32
for i in tqdm(range(0, len(test_texts_m5), BATCH), desc="Weighted ensemble on test.csv"):
    batch_texts = test_texts_m5[i:i + BATCH]
    d_probs = probs_batch(batch_texts, deberta_ft_tokenizer, deberta_ft_model, DEBERTA_MAX_LEN)
    r_probs = probs_batch(batch_texts, roberta_ft_tokenizer, roberta_ft_model, ROBERTA_MAX_LEN)
    w_probs = W_DEBERTA * d_probs + W_ROBERTA * r_probs
    top3_batch = torch.argsort(w_probs, dim=1, descending=True)[:, :3]
    test_top3_preds.extend([' '.join(id_to_label[i.item()] for i in row) for row in top3_batch])

test_df_m5['prediction'] = test_top3_preds
submission_m5 = pd.DataFrame({'id': test_df_m5['id'], 'prediction': test_df_m5['prediction']})
submission_m5.to_csv('submission.csv', index=False)

q5_answer = len(submission_m5)
print(f"Q5 — prediction rows in submission.csv (excluding header): {q5_answer}")
print(submission_m5.head())
print(" ")

# ============================================================
# Q6 — Test-Time Augmentation on the first 50 rows of test.csv
# original prompt vs instruction-augmented prompt, DeBERTa only
# ============================================================
TTA_PREFIX = "Answer the following multiple-choice question carefully: "

first50_texts = test_texts_m5[:50]
first50_aug_texts = [TTA_PREFIX + t for t in first50_texts]

orig_probs_50 = probs_batch(first50_texts, deberta_ft_tokenizer, deberta_ft_model, DEBERTA_MAX_LEN)
aug_probs_50 = probs_batch(first50_aug_texts, deberta_ft_tokenizer, deberta_ft_model, DEBERTA_MAX_LEN)
tta_probs_50 = (orig_probs_50 + aug_probs_50) / 2

orig_top1_50 = orig_probs_50.argmax(dim=1)
tta_top1_50 = tta_probs_50.argmax(dim=1)

q6_answer = int((orig_top1_50 != tta_top1_50).sum())
print(f"Q6 — rows with a different Top-1 prediction after TTA: {q6_answer} / 50")
print(" ")

# ============================================================
# Q7 — first 100 rows: compare Top-1 (DeBERTa alone) vs Top-1 (weighted ensemble)
# ============================================================
first100_texts = test_texts_m5[:100]

deberta_probs_100 = probs_batch(first100_texts, deberta_ft_tokenizer, deberta_ft_model, DEBERTA_MAX_LEN)
roberta_probs_100 = probs_batch(first100_texts, roberta_ft_tokenizer, roberta_ft_model, ROBERTA_MAX_LEN)
weighted_probs_100 = W_DEBERTA * deberta_probs_100 + W_ROBERTA * roberta_probs_100

deberta_top1_100 = deberta_probs_100.argmax(dim=1)
weighted_top1_100 = weighted_probs_100.argmax(dim=1)

q7_answer = int((deberta_top1_100 != weighted_top1_100).sum())
print(f"Q7 — rows with different Top-1 predictions (DeBERTa vs weighted ensemble): {q7_answer} / 100")

# ============================================================
# Q8 — Confidence Gain = Ensemble Confidence - DeBERTa Confidence
# first 100 rows of test.csv
# ============================================================
deberta_conf_100 = deberta_probs_100.max(dim=1).values
weighted_conf_100 = weighted_probs_100.max(dim=1).values
confidence_gain_100 = weighted_conf_100 - deberta_conf_100

q8_answer = int((confidence_gain_100 > 0).sum())
print(f"Q8 — rows with positive confidence gain (> 0): {q8_answer} / 100")
print(" ")

# ============================================================
# Q9 — Top-3 ranking changes: DeBERTa alone vs weighted ensemble
# first 100 rows of test.csv (order matters, e.g. A C D vs A D C)
# ============================================================
deberta_top3_100 = torch.argsort(deberta_probs_100, dim=1, descending=True)[:, :3]
weighted_top3_100 = torch.argsort(weighted_probs_100, dim=1, descending=True)[:, :3]

q9_answer = int((deberta_top3_100 != weighted_top3_100).any(dim=1).sum())
print(f"Q9 — rows with at least one Top-3 ranking change: {q9_answer} / 100")
print(" ")

# ============================================================
# Q10 — MAP@3 of the weighted-ensemble Top-3 predictions
#       for the first 100 validation samples
# ============================================================
val100 = m5_val_split.iloc[:100].reset_index(drop=True)
val100_texts = [format_seqcls(row) for _, row in val100.iterrows()]

val_deberta_probs = probs_batch(val100_texts, deberta_ft_tokenizer, deberta_ft_model, DEBERTA_MAX_LEN)
val_roberta_probs = probs_batch(val100_texts, roberta_ft_tokenizer, roberta_ft_model, ROBERTA_MAX_LEN)
val_weighted_probs = W_DEBERTA * val_deberta_probs + W_ROBERTA * val_roberta_probs

val_top3 = torch.argsort(val_weighted_probs, dim=1, descending=True)[:, :3]
val100_preds = [' '.join(id_to_label[i.item()] for i in row) for row in val_top3]
val100_actuals = val100['answer'].tolist()

q10_answer = round(sum(apk(a, p.split()) for a, p in zip(val100_actuals, val100_preds)) / len(val100_actuals), 4)
print(f"Q10 — Milestone 5 weighted-ensemble MAP@3 (first 100 validation samples): {q10_answer}")